In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, RadioButtons, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# PARAMETERS
# ============================================================

np.random.seed(42)

M_max = 200
N_max = 1000

# ============================================================
# PROCESS 1:
# STATIONARY AND ERGODIC
# ============================================================

X_ergodic = np.random.randn(M_max, N_max)

# ============================================================
# PROCESS 2:
# STATIONARY BUT NON-ERGODIC
# ============================================================

A_values = np.tile([1.0, -1.0], M_max // 2)

X_nonergodic = np.repeat(
    A_values[:, None],
    N_max,
    axis=1
)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_stationary_vs_ergodic(process_type='Stationary & ergodic', M=50, N=300, realization_index=0):

    # --------------------------------------------------------
    # SELECT PROCESS
    # --------------------------------------------------------

    if process_type == 'Stationary & ergodic':

        X = X_ergodic
        process_name = 'Stationary and Ergodic Process'

    else:

        X = X_nonergodic
        process_name = 'Stationary but Non-Ergodic Process'

    # --------------------------------------------------------
    # SELECT ENSEMBLE
    # --------------------------------------------------------

    ensemble = X[:M, :N]

    realization_index = min(realization_index, M - 1)

    realization = ensemble[realization_index, :]

    # --------------------------------------------------------
    # ENSEMBLE MEAN
    # --------------------------------------------------------

    ensemble_mean = np.mean(ensemble[:, -1])

    # --------------------------------------------------------
    # RUNNING TIME MEAN
    # --------------------------------------------------------

    running_time_mean = np.cumsum(realization) / np.arange(1, N + 1)

    final_time_mean = running_time_mean[-1]

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.0, 6.0))

    # ========================================================
    # UPPER GRAPH
    # ========================================================

    ax1.plot(np.arange(N), realization, linewidth=1.2)

    ax1.axhline(
        ensemble_mean,
        color='k',
        linestyle='--',
        linewidth=1.4,
        label=f'Ensemble mean = {ensemble_mean:.3f}'
    )

    ax1.set_xlim(0, N - 1)

    if process_type == 'Stationary & ergodic':
        ax1.set_ylim(-4.0, 4.0)
    else:
        ax1.set_ylim(-1.5, 1.5)

    ax1.set_xlabel('Time index n', fontsize=12)
    ax1.set_ylabel('x[n]', fontsize=12)

    ax1.set_title(
        f'{process_name} - Realization {realization_index}',
        fontsize=13,
        pad=10
    )

    ax1.tick_params(axis='both', labelsize=10)

    ax1.grid(True, linestyle=':', alpha=0.6)

    ax1.legend(fontsize=9, loc='upper right')

    # ========================================================
    # LOWER GRAPH
    # ========================================================

    ax2.plot(
        np.arange(1, N + 1),
        running_time_mean,
        linewidth=2,
        label='Running time mean'
    )

    ax2.axhline(
        ensemble_mean,
        color='k',
        linestyle='--',
        linewidth=1.5,
        label=f'Ensemble mean = {ensemble_mean:.3f}'
    )

    ax2.axhline(
        0,
        linestyle=':',
        linewidth=1.2,
        label='Theoretical ensemble mean = 0'
    )

    ax2.set_xlim(1, N)
    ax2.set_ylim(-1.25, 1.25)

    ax2.set_xlabel('Number of samples N', fontsize=12)
    ax2.set_ylabel('Mean value', fontsize=12)

    ax2.set_title(
        f'Time Mean = {final_time_mean:.3f}     Ensemble Mean = {ensemble_mean:.3f}',
        fontsize=13,
        pad=10
    )

    ax2.tick_params(axis='both', labelsize=10)

    ax2.grid(True, linestyle=':', alpha=0.6)

    ax2.legend(fontsize=9, loc='upper right')

    plt.tight_layout()

    plt.show()

# ============================================================
# RADIO BUTTONS
# ============================================================

process_selector = RadioButtons(
    options=[
        'Stationary & ergodic',
        'Stationary & non-ergodic'
    ],
    value='Stationary & ergodic',
    description='Process:',
    style={'description_width': 'initial'},
    layout=Layout(width='270px')
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

M_slider = IntSlider(
    min=10,
    max=200,
    step=10,
    value=50,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

N_slider = IntSlider(
    min=50,
    max=1000,
    step=50,
    value=300,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

realization_slider = IntSlider(
    min=0,
    max=49,
    step=1,
    value=0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

M_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">200</div>'
)

N_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1000</div>'
)

realization_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">49</div>'
)

# ============================================================
# UPDATE REALIZATION RANGE
# ============================================================

def update_realization_range(change):

    realization_slider.max = M_slider.value - 1

    if realization_slider.value > realization_slider.max:
        realization_slider.value = realization_slider.max

    realization_max_label.value = f'<div style="font-family:Arial; font-size:14px;">{realization_slider.max}</div>'

M_slider.observe(update_realization_range, names='value')

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_stationary_vs_ergodic,
    process_type=process_selector,
    M=M_slider,
    N=N_slider,
    realization_index=realization_slider
)

# ============================================================
# COMPACT THEORY
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
Stationarity Does Not Imply Ergodicity
</div>

<div style="margin-bottom:7px;">
A <b>stationary</b> process has statistical properties that do not change with time.
However, stationarity alone does not guarantee that these statistical properties
can be obtained from a single realization.
</div>

<div style="margin-bottom:7px;">
<b>Stationary and ergodic process:</b> the time average of one sufficiently long
realization approaches the ensemble average.
</div>

<div style="margin-bottom:7px;">
<b>Stationary but non-ergodic process:</b> the process is stationary, but the time
average of a single realization does not approach the ensemble average.
</div>

<div style="margin-bottom:7px;">
In the non-ergodic example, every realization has the form
<b>x[n] = +1</b> for all n or <b>x[n] = -1</b> for all n.
</div>

<div style="margin-bottom:7px;">
The ensemble mean is zero, but the time mean of an individual realization
remains permanently equal to +1 or -1.
</div>

<div>
<b>Therefore: Stationary does not imply ergodic.</b>
</div>

</div>
""")

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

M_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Realizations M:</div>'
)

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Samples N:</div>'
)

realization_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Realization:</div>'
)

# ============================================================
# SLIDER GRID
# ============================================================

slider_grid = GridBox(
    children=[
        M_label, M_slider, M_max_label,
        N_label, N_slider, N_max_label,
        realization_label, realization_slider, realization_max_label
    ],
    layout=Layout(
        width='265px',
        grid_template_columns='110px 100px 40px',
        grid_template_rows='30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# EXTRA SPACE BETWEEN RADIO BUTTONS AND SLIDER GROUP
# ============================================================

slider_group = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        margin='12px 0px 0px 0px'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        process_selector,
        slider_group
    ],
    layout=Layout(
        width='285px',
        min_width='285px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px 10px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1100px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_html,
        graph_and_controls
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)